# NB02b – Simulation Analyse-Zwischendaten
### CAS Information Engineering – Scripting Project (Dev/Helper)

---
Erzeugt alle Ausgabedateien von NB02 unter `sim/processed/` und `sim/intermediate/`.

**Nicht Teil der Einreichung** — nur Entwicklungshilfsmittel.  
Szenario-abhängige Dateien werden in `sim/intermediate/realistisch/` gespeichert.

---
[↑ Projektübersicht](00_Project_Overview.ipynb)


In [1]:
# ── Projektstruktur ────────────────────────────────────────────────────────────
import os, sys, warnings, datetime
import numpy  as np
import pandas as pd
warnings.filterwarnings('ignore')

# Datenquelle: 'data' = echte Downloads | 'sim' = simuliert
MODE = 'data'   # ← hier anpassen: 'data' | 'sim'

DIR_RAW          = os.path.join(MODE, 'raw')
DIR_PROCESSED    = os.path.join(MODE, 'processed')
DIR_INTERMEDIATE = os.path.join(MODE, 'intermediate')
DIR_CHARTS       = 'charts'
DATAINDEX        = 'dataindex.csv'

for d in [DIR_RAW, DIR_PROCESSED, DIR_INTERMEDIATE, DIR_CHARTS,
          os.path.join('data','raw'), os.path.join('data','processed'), os.path.join('data','intermediate'),
          os.path.join('sim', 'raw'), os.path.join('sim', 'processed'), os.path.join('sim', 'intermediate')]:
    os.makedirs(d, exist_ok=True)

print(f'MODE         : {MODE}')
print(f'raw          : {os.path.abspath(DIR_RAW)}')
print(f'processed    : {os.path.abspath(DIR_PROCESSED)}')
print(f'intermediate : {os.path.abspath(DIR_INTERMEDIATE)}')


MODE         : data
raw          : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\raw
processed    : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\processed
intermediate : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\intermediate


In [2]:
# ── dataindex.csv Helper ───────────────────────────────────────────────────────
def log_dataindex(filename, source_url, local_path, data_type,
                  rows=None, size_kb=None, status='active', note=''):
    """
    Zentrales Datenquellen-Register. Jeder Eintrag bleibt als Historie erhalten.
    status: active | superseded | error | partial
    """
    ts = datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z'
    if os.path.exists(DATAINDEX):
        df_idx = pd.read_csv(DATAINDEX)
        mask   = (df_idx['filename'] == filename) & (df_idx['status'] == 'active')
        if mask.any():
            df_idx.loc[mask, 'status']        = 'superseded'
            df_idx.loc[mask, 'superseded_at'] = ts
    else:
        df_idx = pd.DataFrame(columns=['timestamp','filename','source_url','local_path',
                                        'data_type','rows','size_kb','status','superseded_at','note'])
    row = {'timestamp': ts, 'filename': filename, 'source_url': source_url,
           'local_path': local_path, 'data_type': data_type, 'rows': rows,
           'size_kb': round(size_kb,1) if size_kb else None,
           'status': status, 'superseded_at': '', 'note': note}
    pd.concat([df_idx, pd.DataFrame([row])], ignore_index=True).to_csv(DATAINDEX, index=False)
    print(f'  dataindex: {filename} [{status}]')

def log_missing(source, reason):
    ts = datetime.datetime.utcnow().isoformat(timespec='seconds')
    with open(os.path.join('data','missing.txt'), 'a') as f:
        f.write(f'[{ts}] MISSING {source}: {reason}\n')
    log_dataindex(os.path.basename(source), source, '', 'raw', status='error', note=reason)
    print(f'  missing.txt: {reason}')

print('dataindex Helper bereit.')


dataindex Helper bereit.


In [3]:
MODE             = 'sim'
DIR_RAW          = os.path.join(MODE,'raw')
DIR_PROCESSED    = os.path.join(MODE,'processed')
DIR_INTERMEDIATE = os.path.join(MODE,'intermediate')
for d in [DIR_RAW, DIR_PROCESSED, DIR_INTERMEDIATE]: os.makedirs(d, exist_ok=True)
print(f'Zielordner: sim/processed/ und sim/intermediate/')


Zielordner: sim/processed/ und sim/intermediate/


In [4]:
# ── 1. ch_spot_prices_clean.csv → sim/processed/ ──────────────────────────────
CLEAN_FILE = os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv')
RAW_FILE   = os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv')

if os.path.exists(RAW_FILE) and os.path.getsize(RAW_FILE) > 5000:
    df = pd.read_csv(RAW_FILE)
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df['price_eur_mwh'] = pd.to_numeric(df['price_eur_mwh'],errors='coerce').interpolate(limit=3).clip(-500,3000)
else:
    np.random.seed(42)
    idx = pd.date_range('2023-01-01','2024-12-31 23:00',freq='1h',tz='UTC')
    h,m,dow = idx.hour,idx.month,idx.dayofweek
    p = np.clip(60+(-20*np.exp(-((h-4)**2)/2)+25*np.exp(-((h-8.5)**2)/3)
                    -10*np.exp(-((h-14)**2)/4)+30*np.exp(-((h-19)**2)/3))
                +15*np.cos((m-1)*2*np.pi/12)+np.where(dow>=5,-8,0)
                +np.random.normal(0,8,len(idx)),-60,400)
    df = pd.DataFrame({'timestamp':pd.to_datetime(idx),'price_eur_mwh':p.round(2)})

df['timestamp'] = pd.to_datetime(df['timestamp'],utc=True)
for col,fn in [('hour',lambda x:x.dt.hour),('month',lambda x:x.dt.month),
               ('weekday',lambda x:x.dt.dayofweek),
               ('season',lambda x:x.dt.month.map({12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3}))]:
    df[col] = fn(df['timestamp'])
df.to_csv(CLEAN_FILE,index=False)
log_dataindex('ch_spot_prices_clean.csv','NB2b:sim',CLEAN_FILE,'processed/sim',rows=len(df))
print(f'clean.csv → {CLEAN_FILE} | {len(df):,} Zeilen')

# ── 2. wirtschaftlichkeit.csv → sim/intermediate/ ────────────────────────────
df_econ = pd.DataFrame([
    ('Privat_10kWh',    4_000,   92,   60,   32, 125.0, 0.80, 18.4),
    ('Gewerbe_100kWh', 30_000, 1_180,  450,  730,  41.1, 2.43, 23.6),
    ('Industrie_1MWh',220_000,10_200,3_300,6_900,  31.9, 3.14, 20.4),
    ('Utility_10MWh',1_800_000,89_000,27_000,62_000,29.0,3.44,17.8),
],columns=['segment','capex','annual_rev','opex_annual','net_annual',
           'payback_years','roi_pct','rev_per_kwh'])
ECON_FILE = os.path.join(DIR_INTERMEDIATE,'realistisch','wirtschaftlichkeit.csv')
os.makedirs(os.path.join(DIR_INTERMEDIATE,'realistisch'), exist_ok=True)
df_econ.to_csv(ECON_FILE,index=False)
log_dataindex('wirtschaftlichkeit.csv','NB2b:sim',ECON_FILE,'intermediate/sim',rows=4)

# ── 3. netzentlastung_szenarien.csv → sim/intermediate/ ──────────────────────
df_sz = pd.DataFrame([
    ('Status Quo (2024)',       0,     0,    0,    0.0,10.500, 0.00),
    ('Moderat (2027)',     50_000, 2_000,  200,  245.0,10.247, 2.33),
    ('Ambitioniert (2030)',200_000,8_000,  800,  980.0, 9.517, 9.33),
    ('Transformativ (2035)',800_000,30_000,2_000,3920.0,6.577,37.33),
],columns=['szenario','n_privat','n_gewerbe','n_industrie',
           'entlastung_mw','neue_spitzenlast_gw','reduktion_pct'])
SZ_FILE = os.path.join(DIR_INTERMEDIATE,'realistisch','netzentlastung_szenarien.csv')
df_sz.to_csv(SZ_FILE,index=False)
log_dataindex('netzentlastung_szenarien.csv','NB2b:sim',SZ_FILE,'intermediate/sim',rows=4)

print(f'\nNB2b fertig → NB3 mit MODE=\'sim\'')
print(f'  {CLEAN_FILE}')
print(f'  {ECON_FILE}')
print(f'  {SZ_FILE}')


  dataindex: ch_spot_prices_clean.csv [active]
clean.csv → sim\processed\ch_spot_prices_clean.csv | 17,544 Zeilen
  dataindex: wirtschaftlichkeit.csv [active]
  dataindex: netzentlastung_szenarien.csv [active]

NB2b fertig → NB3 mit MODE='sim'
  sim\processed\ch_spot_prices_clean.csv
  sim\intermediate\realistisch\wirtschaftlichkeit.csv
  sim\intermediate\realistisch\netzentlastung_szenarien.csv


In [5]:
# ── Verifikation: Wirtschaftlichkeit (sim) ───────────────────────────────────
df_econ.head()


,segment,capex,annual_rev,opex_annual,net_annual,payback_years,roi_pct,rev_per_kwh
0,Privat_10kWh,4000,92,60,32,125.0,0.80,18.4
1,Gewerbe_100kWh,30000,1180,450,730,41.1,2.43,23.6
2,Industrie_1MWh,220000,10200,3300,6900,31.9,3.14,20.4
3,Utility_10MWh,1800000,89000,27000,62000,29.0,3.44,17.8
